# Predict Yacht resistance

Install welt-client with its parquet extra, then configure WELT_API_KEY locally. Run the cells in order. These are real hosted requests: fitting retains a private training dataset and predictor. No extra helper file is needed. Outputs are deliberately not saved in this source.

We follow six steps: get data → inspect → split → fit → predict → evaluate. Yacht308 rows/six numeric features/recorded residuary resistance target; selected distribution CC0. Seed9 random246/62 teaching split. Related experiments may make random-split scores optimistic; this is not a source benchmark.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from welt import Client, Regressor

# 1. Get a real table from Welt's research inventory.
dataset_id = "asset-bc7dddd884c5e798c1b2abfc559a36ab"
version = "bc7dddd884c5e798c1b2abfc559a36ab79a6d09c4e2ea3e9bbd95c06539c7342"
path = Path(mkdtemp(prefix="welt-yacht-")) / "yacht.parquet"
with Client() as client:
    info = client.research_dataset(dataset_id)
    print("Dataset:", info["name"], "— License:", info["license"])
    for notice in info["license_notices"]:
        print(notice)
    client.download_research_dataset(dataset_id, path, version=version)
data = pd.read_parquet(path)
target = "__target__#0"  # The target recorded in this research snapshot.



In [ ]:
# 2. Look at the rows before modelling.
print(data.head())
print("Rows and columns:", data.shape)
data.info()
print(data[target].describe())



In [ ]:
# 3. Set aside test rows before fitting; never train on their answers.
train, test = train_test_split(
    data, test_size=0.2, random_state=9)
query = test.drop(columns=target)
y_test = test[target]
print("Training rows:", len(train), "— Test rows:", len(test))



In [ ]:
# 4. Create a model and train it on the training table.
model = Regressor(model="tabicl-v2", random_state=9)
model.fit(train, target=target)



In [ ]:
# 5. Predict the held-out rows, without passing their target column.
predictions = model.predict(query)
print(pd.DataFrame({"actual": y_test.to_numpy(), "predicted": predictions}).head())



In [ ]:
# 6. Compare predictions with the answers we kept aside.
print("Test MAE:", mean_absolute_error(y_test, predictions))
print("Test RMSE:", root_mean_squared_error(y_test, predictions))
print("Test R²:", r2_score(y_test, predictions))
print("Keep this predictor ID:", model.predictor_id_)


Read the actual versus predicted rows and the metrics above. Scores depend on your held-out split and model version; no minimum score is promised. Keep the test set for assessment rather than repeated tuning.

In [ ]:
print("Pinned model version:", model.model_version_)
print("Reusable predictor:", model.predictor_id_)